# RSSM Baseline — Training on Colab T4

Recurrent State Space Model (DreamerV2 style, sans actions) — baseline pixel-space à comparer avec **LeWorldModel (JEPA)**.

**Pipeline :**
1. Check GPU
2. Clone repo
3. Dataset (réutilise celui du JEPA/AE si déjà généré)
4. Train RSSM
5. Loss curves
6. Reconstruction visuelle
7. Download checkpoint

**Comparaison des trois approches :**

| | AutoEncoder (AE) | RSSM | LeWorldModel (JEPA) |
|---|---|---|---|
| Supervision | MSE pixel | MSE pixel + KL | Cosine espace latent |
| Dynamique | MLP feedforward | GRU + prior/posterior | MLP feedforward |
| Imagination | Predictor MLP | Prior GRU (sans encodeur) | Predictor MLP |
| Décodeur training | ✓ joint | ✓ joint | ✗ séparé |
| Target encoder EMA | ✗ | ✗ | ✓ |
| Coût énergétique | moyen | élevé (GRU séquentiel) | faible |

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 3. Install dependencies

In [ ]:
!pip install -q Pillow matplotlib numpy
print('Dependencies ready.')

## 4. Dataset

Réutilise le dataset généré pour l'AE/JEPA si disponible.  
Sinon génère **2 000 trajectoires × 500 frames** (64×64 px).

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from generate_dataset import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `feat_dim` | 128 | sortie encodeur CNN — identique AE/JEPA pour comparaison équitable |
| `h_dim` | 200 | état déterministe GRU (PlaNet/DreamerV2 default) |
| `s_dim` | 32 | état stochastique |
| `hidden_dim` | 256 | MLP prior/posterior |
| `kl_scale` | 1.0 | poids du terme KL dans la loss |
| `free_nats` | 1.0 | plancher KL/pas (évite sur-pénalisation initiale) |
| `pixel_weight` | 10 | sur-pondération pixels brillants dans wmse |
| `seq_len` | 50 | fenêtre (plus courte qu'AE car GRU séquentiel = plus lent) |
| `epochs` | 100 | |
| `batch_size` | 32 | |
| `lr` | 6e-4 | AdamW (DreamerV2 default) |

**Coût comparatif :**  
Le RSSM est plus lent à l'entraînement que l'AE (boucle GRU séquentielle sur T=50 pas)  
et que le JEPA (pas de décodeur pixel pendant l'entraînement).  
C'est le cœur de la comparaison énergétique avec JEPA.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ── Hyperparameters ────────────────────────────────────────────────────────────
FEAT_DIM     = 128   # sortie encodeur CNN (= embed_dim AE/JEPA)
H_DIM        = 200   # état déterministe GRU
S_DIM        = 32    # état stochastique
HIDDEN_DIM   = 256   # MLP prior/posterior
KL_SCALE     = 0.0   # 0 = GRU déterministe pur, pas de KL floor qui masque le gradient
FREE_NATS    = 0.0
SEQ_LEN      = 50
EPOCHS       = 100
BATCH_SIZE   = 32
LR           = 6e-4
WEIGHT_DECAY = 1e-6
PIXEL_WEIGHT = 50.0  # 50 → bras blanc (2% pixels) contribue à ~51% de la loss
                     # vs 18% avec pixel_weight=10 — signal fort sur la dynamique du bras
NUM_WORKERS  = 2
CKPT_DIR     = 'checkpoints'
VIS_DIR      = 'visuals'
RESUME       = None  # set to 'checkpoints/rssm_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.rssm import RSSM
from dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = RSSM(
    feat_dim=FEAT_DIM,
    h_dim=H_DIM,
    s_dim=S_DIM,
    hidden_dim=HIDDEN_DIM,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')
print(f'Latent dim: h={H_DIM}  s={S_DIM}  total={H_DIM + S_DIM}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate — free_nats identique au train pour des losses comparables ────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total_recon = total_kl = total_kl_raw = 0.0
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True),
                  kl_scale=KL_SCALE, pixel_weight=PIXEL_WEIGHT, free_nats=FREE_NATS)
        total_loss    += m['loss'].item()
        total_recon   += m['recon_loss'].item()
        total_kl      += m['kl_loss'].item()
        total_kl_raw  += m['kl_raw'].item()
    n = len(loader)
    return {'loss': total_loss/n, 'recon_loss': total_recon/n,
            'kl_loss': total_kl/n, 'kl_raw': total_kl_raw/n}

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

torch.cuda.reset_peak_memory_stats(device)

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history  = {'train': [], 'val': [], 'kl_raw': [], 'epoch_time': []}
best_val = float('inf')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0         = time.time()
    total_loss = total_kl_raw = 0.0

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames, kl_scale=KL_SCALE,
                  pixel_weight=PIXEL_WEIGHT, free_nats=FREE_NATS)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        optimizer.step()
        total_loss    += m['loss'].item()
        total_kl_raw  += m['kl_raw'].item()

    scheduler.step()
    elapsed    = time.time() - t0
    n          = len(train_loader)
    train_loss = total_loss / n
    kl_raw     = total_kl_raw / n
    val_m      = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_m['loss'])
    history['kl_raw'].append(kl_raw)
    history['epoch_time'].append(elapsed)

    improved = val_m['loss'] < best_val
    if improved:
        best_val = val_m['loss']
        torch.save({
            'epoch':    epoch,
            'model':    model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss': best_val,
            'args': {
                'feat_dim':     FEAT_DIM,
                'h_dim':        H_DIM,
                's_dim':        S_DIM,
                'hidden_dim':   HIDDEN_DIM,
                'kl_scale':     KL_SCALE,
                'free_nats':    FREE_NATS,
                'pixel_weight': PIXEL_WEIGHT,
            },
        }, f'{CKPT_DIR}/rssm_best.pt')

    lr_now = optimizer.param_groups[0]['lr']
    clamped = kl_raw < FREE_NATS
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  loss={train_loss:.5f}'
        f'  kl={kl_raw:.3f}{"*" if clamped else ""}'   # * = posterior collapse (kl < free_nats)
        f'  val={val_m["loss"]:.5f}'
        f'  lr={lr_now:.2e}'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

mem_mb = torch.cuda.max_memory_allocated(device) / 1e6
avg_t  = sum(history['epoch_time']) / len(history['epoch_time'])
print(f'\nBest val loss       : {best_val:.6f}  ->  {CKPT_DIR}/rssm_best.pt')
print(f'Temps moyen / epoch : {avg_t:.1f}s')
print(f'Pic mémoire GPU     : {mem_mb:.0f} MB')
print(f'\nNote: kl* = KL brute < free_nats → gradient KL nul, GRU fait tout le travail')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#111')

panels = [
    (axes[0], 'Loss totale (recon + kl_scale*KL)',
     history['train'], history['val'], '#4fc3f7', '#ff8a65', 'train', 'val'),
    (axes[1], 'KL loss (train)',
     history['kl_loss'], None, '#a5d6a7', None, 'kl_loss', None),
    (axes[2], 'Temps par epoch (s)',
     history['epoch_time'], None, '#ce93d8', None, 'temps/epoch', None),
]

for ax, title, d1, d2, c1, c2, l1, l2 in panels:
    ax.set_facecolor('#111')
    ax.plot(epochs_x, d1, color=c1, label=l1)
    if d2 is not None:
        ax.plot(epochs_x, d2, color=c2, label=l2)
    ax.set_title(title, color='white')
    ax.set_xlabel('epoch', color='white')
    ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/rssm_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Reconstruction visuelle

Compare les frames reconstruites par le RSSM avec les frames réelles.

- **Lignes paires** : frames reconstruites (RSSM)
- **Lignes impaires** : frames réelles
- **Colonnes** : différents pas de temps (t=0 à t=4)

In [ ]:
import torch.nn.functional as F

model.eval()

frames_batch, _ = next(iter(val_loader))
frames_batch = frames_batch.to(device)   # (B, T, 3, 64, 64)
B, T, C, H, W = frames_batch.shape

with torch.no_grad():
    # encode() retourne h_seq uniquement (h_dim=200), cohérent avec le décodeur
    h_seq  = model.encode(frames_batch)          # (B, T, h_dim)
    recon  = model.decoder(h_seq)                # (B, T, 3, H, W)
    recon_mse = F.mse_loss(recon, frames_batch).item()

n_show  = min(4, B)
n_steps = min(5, T)

fig, axes = plt.subplots(n_show * 2, n_steps, figsize=(n_steps * 2, n_show * 4 + 0.5))
fig.patch.set_facecolor('#111')

for row, b in enumerate(range(n_show)):
    for col in range(n_steps):
        pred_img = recon[b, col].permute(1, 2, 0).cpu().numpy()
        real_img = frames_batch[b, col].permute(1, 2, 0).cpu().numpy()

        ax_pred = axes[row * 2,     col]
        ax_real = axes[row * 2 + 1, col]

        ax_pred.imshow(pred_img.clip(0, 1))
        ax_real.imshow(real_img.clip(0, 1))
        ax_pred.axis('off')
        ax_real.axis('off')

        if row == 0:
            ax_pred.set_title(f't={col}', color='white', fontsize=9)
        if col == 0:
            ax_pred.set_ylabel('prédit', color='#a5d6a7', fontsize=8)
            ax_real.set_ylabel('réel',   color='#4fc3f7', fontsize=8)

fig.suptitle(f'Reconstruction RSSM  —  MSE={recon_mse:.5f}  (décodeur depuis h_t uniquement)',
             color='white', fontsize=11)
plt.tight_layout()
plt.savefig(f'{VIS_DIR}/rssm_reconstruction.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

## 8. Résumé énergétique

Compare le coût d'entraînement des trois approches.

In [ ]:
avg_epoch_s  = sum(history['epoch_time']) / len(history['epoch_time'])
total_time_s = sum(history['epoch_time'])

print('── Coût RSSM ──────────────────────────────────────')
print(f'  Temps moyen / epoch : {avg_epoch_s:.1f} s')
print(f'  Temps total         : {total_time_s / 60:.1f} min  ({EPOCHS} epochs)')
print(f'  Pic mémoire GPU     : {mem_mb:.0f} MB')
print(f'  Latent dim          : {H_DIM + S_DIM}  (h={H_DIM}, s={S_DIM})')
print()
print('  Pour comparer avec AE et JEPA :')
print('  → noter temps/epoch ici et dans les notebooks AE/JEPA')
print('  → JEPA devrait être le plus rapide (pas de décodeur pixel)')

## 9. Download checkpoint

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/rssm_best.pt')